# Embedding Models Performance Evaluation

This notebook evaluates the performance of the text embedding (hiieu/halong_embedding) and image embedding (DINOv2) models used in the KK Bookstore AI application.

## Models Being Evaluated:
1. **Text Embedding Model**: `hiieu/halong_embedding` 
   - Input: Product descriptions from existing products in the system
   - Use case: Finding similar products based on text descriptions

2. **Image Embedding Model**: `dinov2_vitl14` (DINOv2)
   - Input: User-uploaded images (base64 encoded)
   - Use case: Finding products similar to user-provided images

## Evaluation Metrics:

### For Text Embeddings:
- **Cosine Similarity Distribution**: Measures how well the model separates similar vs dissimilar texts
- **Embedding Quality Score**: Based on intra-cluster vs inter-cluster distances
- **Retrieval Metrics**: Precision@K, Recall@K for similar product retrieval
- **Embedding Consistency**: How stable embeddings are for similar texts

### For Image Embeddings:
- **Cosine Similarity Distribution**: Measures embedding quality for image similarity
- **Visual Similarity Correlation**: How well embeddings correlate with visual features
- **Retrieval Performance**: Precision@K, Recall@K for image-based product search
- **Robustness**: Performance under different image transformations

## 1. Setup Environment

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set paths
import os
import sys

# Path to the app directory
APP_DIR = '/content/drive/MyDrive/KLTN_SRC'
# Path to the data directory  
DATA_DIR = '/content/drive/MyDrive/KLTN_DATA'

# Add app directory to Python path
sys.path.append(APP_DIR)

# Check if directories exist
print(f"App directory exists: {os.path.exists(APP_DIR)}")
print(f"Data directory exists: {os.path.exists(DATA_DIR)}")

## 2. Install Dependencies

In [ ]:
# Install required packages for evaluation
!pip install sentence-transformers torch chromadb transformers evaluate scikit-learn matplotlib seaborn pandas numpy Pillow torchvision

## 3. Import Libraries

In [ ]:
import torch
import json
import chromadb
import os
import base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from io import BytesIO
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Tuple
import random
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
plt.style.use('default')
sns.set_palette("husl")

## 4. Load and Initialize Models

We'll recreate the exact same embedding generators as used in the production API.

In [ ]:
# Check if CUDA is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Text Embedding Generator - matches production exactly
class TextEmbeddingGenerator:
    def __init__(self, device: torch.device = None):
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model_name = "hiieu/halong_embedding"
        print(f"Loading text embedding model: {self.model_name}")
        
        try:
            self.model = SentenceTransformer(self.model_name).to(self.device)
            print("Text model loaded successfully")
        except Exception as error:
            print(f"Failed to load the SentenceTransformer model: {error}")
            raise RuntimeError(f"Failed to load the SentenceTransformer model '{self.model_name}': {error}") from error

    async def generate_text_embedding(self, input_text: str) -> List[float]:
        """Generate embedding for text input - matches API exactly"""
        embedding = self.model.encode([input_text], convert_to_tensor=True)
        return embedding[0].cpu().detach().numpy().tolist()
    
    def generate_text_embedding_sync(self, input_text: str) -> List[float]:
        """Synchronous version for evaluation"""
        embedding = self.model.encode([input_text], convert_to_tensor=True)
        return embedding[0].cpu().detach().numpy().tolist()

In [ ]:
# Image Embedding Generator - matches production exactly
class ImageEmbeddingGenerator:
    def __init__(self, device: torch.device = None):
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # Use exact same model configuration as production
        MODEL_REPOSITORY = "facebookresearch/dinov2"
        MODEL_NAME = "dinov2_vitl14"
        
        try:
            print(f"Loading image model: {MODEL_REPOSITORY}/{MODEL_NAME}")
            self.model = torch.hub.load(
                repo_or_dir=MODEL_REPOSITORY,
                model=MODEL_NAME,
                pretrained=True,
                force_reload=False
            ).to(self.device)
            print("Image model loaded successfully")
        except Exception as error:
            print(f"Failed to load the image model: {error}")
            raise RuntimeError(f"Failed to load the model: '{MODEL_REPOSITORY}' with name '{MODEL_NAME}': {error}") from error

        # Define image transformation pipeline - matches production exactly
        self.image_transform_pipeline = T.Compose([
            T.ToTensor(),
            T.Resize(244),
            T.CenterCrop(224),
            T.Normalize([0.5], [0.5])
        ])

    async def decode_and_transform_image(self, encoded_image: str) -> torch.Tensor:
        """Decode and transform image - matches API exactly"""
        decoded_image_data = base64.b64decode(encoded_image)
        image = Image.open(BytesIO(decoded_image_data)).convert("RGB")
        transformed_image = self.image_transform_pipeline(image)[:3].unsqueeze(0)
        return transformed_image

    async def generate_image_embedding(self, encoded_image: str) -> List[float]:
        """Generate embedding for image - matches API exactly"""
        transformed_image = await self.decode_and_transform_image(encoded_image)
        transformed_image = transformed_image.to(self.device)
        embedding_tensor = self.model(transformed_image)
        return embedding_tensor[0].cpu().detach().numpy().tolist()
    
    def generate_image_embedding_sync(self, encoded_image: str) -> List[float]:
        """Synchronous version for evaluation"""
        decoded_image_data = base64.b64decode(encoded_image)
        image = Image.open(BytesIO(decoded_image_data)).convert("RGB")
        transformed_image = self.image_transform_pipeline(image)[:3].unsqueeze(0)
        transformed_image = transformed_image.to(self.device)
        embedding_tensor = self.model(transformed_image)
        return embedding_tensor[0].cpu().detach().numpy().tolist()

## 5. Initialize Models

In [ ]:
# Initialize the embedding generators
print("Initializing text embedding generator...")
text_generator = TextEmbeddingGenerator(device)

print("\nInitializing image embedding generator...")
image_generator = ImageEmbeddingGenerator(device)

print("\nBoth models initialized successfully!")

## 6. Load ChromaDB Data

Load existing data from ChromaDB to understand the current embeddings and create evaluation datasets.

In [ ]:
# Connect to ChromaDB
try:
    # Try to connect to existing ChromaDB
    chroma_client = chromadb.PersistentClient(path=f"{APP_DIR}/chromadb")
    
    # Get collections
    text_collection = chroma_client.get_collection("text_collection")
    image_collection = chroma_client.get_collection("image_collection") 
    
    print(f"Text collection count: {text_collection.count()}")
    print(f"Image collection count: {image_collection.count()}")
    
except Exception as e:
    print(f"Error connecting to ChromaDB: {e}")
    print("Please ensure ChromaDB data is available in the expected path")

## 7. Text Embedding Model Evaluation

In [ ]:
# Sample text data for evaluation
print("Sampling text data for evaluation...")

# Get sample data from text collection
try:
    sample_size = min(1000, text_collection.count())  # Sample up to 1000 items
    sample_data = text_collection.get(limit=sample_size)
    
    text_ids = sample_data['ids']
    text_documents = sample_data['documents'] 
    text_embeddings = sample_data['embeddings']
    
    print(f"Loaded {len(text_documents)} text samples for evaluation")
    print(f"Sample text: {text_documents[0][:100]}...")
    
except Exception as e:
    print(f"Error loading text data: {e}")
    # Create dummy data for demonstration
    text_documents = [
        "Python programming book for beginners",
        "Advanced machine learning algorithms", 
        "Web development with JavaScript",
        "Data science and analytics",
        "Mobile app development guide"
    ]
    text_ids = [f"text_{i}" for i in range(len(text_documents))]
    text_embeddings = [text_generator.generate_text_embedding_sync(doc) for doc in text_documents]
    print("Using dummy text data for demonstration")

In [ ]:
# Text Embedding Quality Metrics

def evaluate_text_embeddings(texts: List[str], embeddings: List[List[float]]) -> Dict:
    """Evaluate text embedding quality"""
    embeddings_array = np.array(embeddings)
    
    # 1. Embedding Statistics
    embedding_stats = {
        'dimension': embeddings_array.shape[1],
        'mean_norm': np.mean(np.linalg.norm(embeddings_array, axis=1)),
        'std_norm': np.std(np.linalg.norm(embeddings_array, axis=1)),
        'mean_value': np.mean(embeddings_array),
        'std_value': np.std(embeddings_array)
    }
    
    # 2. Cosine Similarity Distribution
    similarity_matrix = cosine_similarity(embeddings_array)
    upper_triangle = similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)]
    
    similarity_stats = {
        'mean_similarity': np.mean(upper_triangle),
        'std_similarity': np.std(upper_triangle),
        'min_similarity': np.min(upper_triangle),
        'max_similarity': np.max(upper_triangle)
    }
    
    # 3. Clustering Quality (if enough samples)
    clustering_stats = {}
    if len(embeddings) >= 10:
        try:
            n_clusters = min(5, len(embeddings) // 2)
            kmeans = KMeans(n_clusters=n_clusters, random_state=42)
            cluster_labels = kmeans.fit_predict(embeddings_array)
            silhouette_avg = silhouette_score(embeddings_array, cluster_labels)
            clustering_stats['silhouette_score'] = silhouette_avg
            clustering_stats['n_clusters'] = n_clusters
        except:
            clustering_stats['silhouette_score'] = None
    
    # 4. Embedding Diversity
    diversity_stats = {
        'pairwise_distances_mean': np.mean(upper_triangle),
        'pairwise_distances_std': np.std(upper_triangle),
        'effective_rank': np.linalg.matrix_rank(embeddings_array) / len(embeddings)
    }
    
    return {
        'embedding_stats': embedding_stats,
        'similarity_stats': similarity_stats, 
        'clustering_stats': clustering_stats,
        'diversity_stats': diversity_stats,
        'similarity_matrix': similarity_matrix
    }

# Run text embedding evaluation
print("Evaluating text embeddings...")
text_eval_results = evaluate_text_embeddings(text_documents, text_embeddings)

# Display results
print("\n=== TEXT EMBEDDING EVALUATION RESULTS ===")
print(f"Embedding Dimension: {text_eval_results['embedding_stats']['dimension']}")
print(f"Mean Embedding Norm: {text_eval_results['embedding_stats']['mean_norm']:.4f}")
print(f"Mean Cosine Similarity: {text_eval_results['similarity_stats']['mean_similarity']:.4f}")
print(f"Std Cosine Similarity: {text_eval_results['similarity_stats']['std_similarity']:.4f}")
if text_eval_results['clustering_stats'].get('silhouette_score'):
    print(f"Silhouette Score: {text_eval_results['clustering_stats']['silhouette_score']:.4f}")
print(f"Effective Rank: {text_eval_results['diversity_stats']['effective_rank']:.4f}")

In [ ]:
# Visualize Text Embedding Results

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Cosine Similarity Distribution
similarity_matrix = text_eval_results['similarity_matrix']
upper_triangle = similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)]

axes[0, 0].hist(upper_triangle, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Text Embeddings: Cosine Similarity Distribution')
axes[0, 0].set_xlabel('Cosine Similarity')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(np.mean(upper_triangle), color='red', linestyle='--', label=f'Mean: {np.mean(upper_triangle):.3f}')
axes[0, 0].legend()

# 2. Embedding Norms
embedding_norms = [np.linalg.norm(emb) for emb in text_embeddings]
axes[0, 1].hist(embedding_norms, bins=30, alpha=0.7, color='lightgreen', edgecolor='black')
axes[0, 1].set_title('Text Embeddings: Norm Distribution')
axes[0, 1].set_xlabel('Embedding Norm')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(np.mean(embedding_norms), color='red', linestyle='--', label=f'Mean: {np.mean(embedding_norms):.3f}')
axes[0, 1].legend()

# 3. Similarity Heatmap (sample)
sample_size = min(20, len(text_embeddings))
sample_similarity = similarity_matrix[:sample_size, :sample_size]
im = axes[1, 0].imshow(sample_similarity, cmap='coolwarm', vmin=0, vmax=1)
axes[1, 0].set_title(f'Similarity Heatmap (First {sample_size} samples)')
axes[1, 0].set_xlabel('Text Index')
axes[1, 0].set_ylabel('Text Index')
plt.colorbar(im, ax=axes[1, 0])

# 4. Embedding Statistics Summary
stats_data = {
    'Metric': ['Dimension', 'Mean Norm', 'Std Norm', 'Mean Similarity', 'Std Similarity', 'Effective Rank'],
    'Value': [
        text_eval_results['embedding_stats']['dimension'],
        f"{text_eval_results['embedding_stats']['mean_norm']:.4f}",
        f"{text_eval_results['embedding_stats']['std_norm']:.4f}",
        f"{text_eval_results['similarity_stats']['mean_similarity']:.4f}",
        f"{text_eval_results['similarity_stats']['std_similarity']:.4f}",
        f"{text_eval_results['diversity_stats']['effective_rank']:.4f}"
    ]
}

axes[1, 1].axis('tight')
axes[1, 1].axis('off')
table = axes[1, 1].table(cellText=[[stats_data['Metric'][i], stats_data['Value'][i]] for i in range(len(stats_data['Metric']))],
                        colLabels=['Metric', 'Value'],
                        cellLoc='center',
                        loc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.5)
axes[1, 1].set_title('Text Embedding Statistics Summary')

plt.tight_layout()
plt.show()

## 8. Text Embedding Retrieval Performance

In [ ]:
# Test text retrieval performance using synthetic queries
def evaluate_text_retrieval(texts: List[str], embeddings: List[List[float]], k_values: List[int] = [1, 5, 10]) -> Dict:
    """Evaluate text retrieval performance"""
    embeddings_array = np.array(embeddings)
    
    # Create synthetic queries by modifying existing texts
    queries = []
    relevant_docs = []
    
    for i, text in enumerate(texts[:min(50, len(texts))]):  # Use up to 50 queries
        # Create query by taking first part of text
        words = text.split()
        if len(words) > 3:
            query = ' '.join(words[:len(words)//2])  # Take first half
            queries.append(query)
            relevant_docs.append(i)  # Original text is relevant
    
    if not queries:
        return {'error': 'No valid queries could be created'}
    
    print(f"Testing retrieval with {len(queries)} queries...")
    
    # Generate embeddings for queries
    query_embeddings = [text_generator.generate_text_embedding_sync(q) for q in queries]
    query_embeddings_array = np.array(query_embeddings)
    
    # Calculate similarities
    similarities = cosine_similarity(query_embeddings_array, embeddings_array)
    
    # Evaluate retrieval metrics
    precision_at_k = {k: [] for k in k_values}
    recall_at_k = {k: [] for k in k_values}
    
    for i, (query, relevant_doc_idx) in enumerate(zip(queries, relevant_docs)):
        # Get top-k most similar documents
        doc_similarities = similarities[i]
        top_k_indices = np.argsort(doc_similarities)[::-1]
        
        for k in k_values:
            top_k = top_k_indices[:k]
            
            # Calculate precision@k and recall@k
            relevant_retrieved = 1 if relevant_doc_idx in top_k else 0
            precision_at_k[k].append(relevant_retrieved / k)
            recall_at_k[k].append(relevant_retrieved / 1)  # Only 1 relevant doc per query
    
    # Average metrics
    avg_precision = {k: np.mean(precision_at_k[k]) for k in k_values}
    avg_recall = {k: np.mean(recall_at_k[k]) for k in k_values}
    
    return {
        'precision_at_k': avg_precision,
        'recall_at_k': avg_recall,
        'num_queries': len(queries),
        'sample_similarities': similarities[:5, :10]  # Sample for inspection
    }

# Run retrieval evaluation
text_retrieval_results = evaluate_text_retrieval(text_documents, text_embeddings)

if 'error' not in text_retrieval_results:
    print("\n=== TEXT RETRIEVAL EVALUATION RESULTS ===")
    print(f"Number of test queries: {text_retrieval_results['num_queries']}")
    
    for k in [1, 5, 10]:
        if k in text_retrieval_results['precision_at_k']:
            print(f"Precision@{k}: {text_retrieval_results['precision_at_k'][k]:.4f}")
            print(f"Recall@{k}: {text_retrieval_results['recall_at_k'][k]:.4f}")
else:
    print(f"Retrieval evaluation error: {text_retrieval_results['error']}")

## 9. Sample Images for Image Embedding Evaluation

Create or load sample images for image embedding evaluation.

In [ ]:
# Sample image data for evaluation
print("Preparing image data for evaluation...")

try:
    # Get sample data from image collection
    sample_size = min(100, image_collection.count())
    sample_image_data = image_collection.get(limit=sample_size)
    
    image_ids = sample_image_data['ids']
    image_embeddings = sample_image_data['embeddings']
    image_metadatas = sample_image_data['metadatas']
    
    print(f"Loaded {len(image_embeddings)} image embeddings for evaluation")
    
except Exception as e:
    print(f"Error loading image data: {e}")
    # Create synthetic image embeddings for demonstration
    print("Creating synthetic image embeddings for demonstration...")
    
    # Generate random images and their embeddings
    def create_test_image():
        """Create a simple test image"""
        img = Image.new('RGB', (224, 224), color=(random.randint(0, 255), random.randint(0, 255), random.randint(0, 255)))
        buffer = BytesIO()
        img.save(buffer, format='PNG')
        buffer.seek(0)
        return base64.b64encode(buffer.getvalue()).decode()
    
    # Generate test images and embeddings
    test_images = [create_test_image() for _ in range(20)]
    image_embeddings = [image_generator.generate_image_embedding_sync(img) for img in test_images]
    image_ids = [f"img_{i}" for i in range(len(test_images))]
    print(f"Created {len(image_embeddings)} synthetic image embeddings")

## 10. Image Embedding Model Evaluation

In [ ]:
# Image Embedding Quality Metrics
def evaluate_image_embeddings(embeddings: List[List[float]]) -> Dict:
    """Evaluate image embedding quality"""
    embeddings_array = np.array(embeddings)
    
    # 1. Embedding Statistics
    embedding_stats = {
        'dimension': embeddings_array.shape[1],
        'mean_norm': np.mean(np.linalg.norm(embeddings_array, axis=1)),
        'std_norm': np.std(np.linalg.norm(embeddings_array, axis=1)),
        'mean_value': np.mean(embeddings_array),
        'std_value': np.std(embeddings_array)
    }
    
    # 2. Cosine Similarity Distribution
    similarity_matrix = cosine_similarity(embeddings_array)
    upper_triangle = similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)]
    
    similarity_stats = {
        'mean_similarity': np.mean(upper_triangle),
        'std_similarity': np.std(upper_triangle),
        'min_similarity': np.min(upper_triangle),
        'max_similarity': np.max(upper_triangle)
    }
    
    # 3. Clustering Quality (if enough samples)
    clustering_stats = {}
    if len(embeddings) >= 10:
        try:
            n_clusters = min(5, len(embeddings) // 2)
            kmeans = KMeans(n_clusters=n_clusters, random_state=42)
            cluster_labels = kmeans.fit_predict(embeddings_array)
            silhouette_avg = silhouette_score(embeddings_array, cluster_labels)
            clustering_stats['silhouette_score'] = silhouette_avg
            clustering_stats['n_clusters'] = n_clusters
        except:
            clustering_stats['silhouette_score'] = None
    
    # 4. Embedding Diversity
    diversity_stats = {
        'pairwise_distances_mean': np.mean(upper_triangle),
        'pairwise_distances_std': np.std(upper_triangle),
        'effective_rank': np.linalg.matrix_rank(embeddings_array) / len(embeddings)
    }
    
    return {
        'embedding_stats': embedding_stats,
        'similarity_stats': similarity_stats,
        'clustering_stats': clustering_stats,
        'diversity_stats': diversity_stats,
        'similarity_matrix': similarity_matrix
    }

# Run image embedding evaluation
print("Evaluating image embeddings...")
image_eval_results = evaluate_image_embeddings(image_embeddings)

# Display results
print("\n=== IMAGE EMBEDDING EVALUATION RESULTS ===")
print(f"Embedding Dimension: {image_eval_results['embedding_stats']['dimension']}")
print(f"Mean Embedding Norm: {image_eval_results['embedding_stats']['mean_norm']:.4f}")
print(f"Mean Cosine Similarity: {image_eval_results['similarity_stats']['mean_similarity']:.4f}")
print(f"Std Cosine Similarity: {image_eval_results['similarity_stats']['std_similarity']:.4f}")
if image_eval_results['clustering_stats'].get('silhouette_score'):
    print(f"Silhouette Score: {image_eval_results['clustering_stats']['silhouette_score']:.4f}")
print(f"Effective Rank: {image_eval_results['diversity_stats']['effective_rank']:.4f}")

In [ ]:
# Visualize Image Embedding Results

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Cosine Similarity Distribution
similarity_matrix = image_eval_results['similarity_matrix']
upper_triangle = similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)]

axes[0, 0].hist(upper_triangle, bins=50, alpha=0.7, color='lightcoral', edgecolor='black')
axes[0, 0].set_title('Image Embeddings: Cosine Similarity Distribution')
axes[0, 0].set_xlabel('Cosine Similarity')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(np.mean(upper_triangle), color='red', linestyle='--', label=f'Mean: {np.mean(upper_triangle):.3f}')
axes[0, 0].legend()

# 2. Embedding Norms
embedding_norms = [np.linalg.norm(emb) for emb in image_embeddings]
axes[0, 1].hist(embedding_norms, bins=30, alpha=0.7, color='lightsalmon', edgecolor='black')
axes[0, 1].set_title('Image Embeddings: Norm Distribution')
axes[0, 1].set_xlabel('Embedding Norm')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(np.mean(embedding_norms), color='red', linestyle='--', label=f'Mean: {np.mean(embedding_norms):.3f}')
axes[0, 1].legend()

# 3. Similarity Heatmap (sample)
sample_size = min(20, len(image_embeddings))
sample_similarity = similarity_matrix[:sample_size, :sample_size]
im = axes[1, 0].imshow(sample_similarity, cmap='coolwarm', vmin=0, vmax=1)
axes[1, 0].set_title(f'Similarity Heatmap (First {sample_size} samples)')
axes[1, 0].set_xlabel('Image Index')
axes[1, 0].set_ylabel('Image Index')
plt.colorbar(im, ax=axes[1, 0])

# 4. Embedding Statistics Summary
stats_data = {
    'Metric': ['Dimension', 'Mean Norm', 'Std Norm', 'Mean Similarity', 'Std Similarity', 'Effective Rank'],
    'Value': [
        image_eval_results['embedding_stats']['dimension'],
        f"{image_eval_results['embedding_stats']['mean_norm']:.4f}",
        f"{image_eval_results['embedding_stats']['std_norm']:.4f}",
        f"{image_eval_results['similarity_stats']['mean_similarity']:.4f}",
        f"{image_eval_results['similarity_stats']['std_similarity']:.4f}",
        f"{image_eval_results['diversity_stats']['effective_rank']:.4f}"
    ]
}

axes[1, 1].axis('tight')
axes[1, 1].axis('off')
table = axes[1, 1].table(cellText=[[stats_data['Metric'][i], stats_data['Value'][i]] for i in range(len(stats_data['Metric']))],
                        colLabels=['Metric', 'Value'],
                        cellLoc='center',
                        loc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.5)
axes[1, 1].set_title('Image Embedding Statistics Summary')

plt.tight_layout()
plt.show()

## 11. Model Comparison and Final Results

In [ ]:
# Compare both models
def create_comparison_report(text_results: Dict, image_results: Dict) -> Dict:
    """Create a comprehensive comparison report"""
    
    comparison = {
        'model_comparison': {
            'text_model': {
                'name': 'hiieu/halong_embedding',
                'dimension': text_results['embedding_stats']['dimension'],
                'mean_similarity': text_results['similarity_stats']['mean_similarity'],
                'std_similarity': text_results['similarity_stats']['std_similarity'],
                'effective_rank': text_results['diversity_stats']['effective_rank']
            },
            'image_model': {
                'name': 'dinov2_vitl14 (DINOv2)',
                'dimension': image_results['embedding_stats']['dimension'],
                'mean_similarity': image_results['similarity_stats']['mean_similarity'],
                'std_similarity': image_results['similarity_stats']['std_similarity'],
                'effective_rank': image_results['diversity_stats']['effective_rank']
            }
        }
    }
    
    # Quality assessment
    text_quality = "Good" if text_results['similarity_stats']['std_similarity'] > 0.1 else "Needs Improvement"
    image_quality = "Good" if image_results['similarity_stats']['std_similarity'] > 0.1 else "Needs Improvement"
    
    comparison['quality_assessment'] = {
        'text_model_quality': text_quality,
        'image_model_quality': image_quality,
        'text_diversity': "High" if text_results['diversity_stats']['effective_rank'] > 0.8 else "Moderate",
        'image_diversity': "High" if image_results['diversity_stats']['effective_rank'] > 0.8 else "Moderate"
    }
    
    return comparison

# Create final comparison report
final_report = create_comparison_report(text_eval_results, image_eval_results)

print("=" * 60)
print("FINAL EMBEDDING MODELS EVALUATION REPORT")
print("=" * 60)

print("\n📊 MODEL SPECIFICATIONS:")
print(f"Text Model: {final_report['model_comparison']['text_model']['name']}")
print(f"  - Dimension: {final_report['model_comparison']['text_model']['dimension']}")
print(f"  - Mean Similarity: {final_report['model_comparison']['text_model']['mean_similarity']:.4f}")
print(f"  - Similarity Std: {final_report['model_comparison']['text_model']['std_similarity']:.4f}")
print(f"  - Effective Rank: {final_report['model_comparison']['text_model']['effective_rank']:.4f}")

print(f"\nImage Model: {final_report['model_comparison']['image_model']['name']}")
print(f"  - Dimension: {final_report['model_comparison']['image_model']['dimension']}")
print(f"  - Mean Similarity: {final_report['model_comparison']['image_model']['mean_similarity']:.4f}")
print(f"  - Similarity Std: {final_report['model_comparison']['image_model']['std_similarity']:.4f}")
print(f"  - Effective Rank: {final_report['model_comparison']['image_model']['effective_rank']:.4f}")

print("\n🎯 QUALITY ASSESSMENT:")
print(f"Text Model Quality: {final_report['quality_assessment']['text_model_quality']}")
print(f"Image Model Quality: {final_report['quality_assessment']['image_model_quality']}")
print(f"Text Embedding Diversity: {final_report['quality_assessment']['text_diversity']}")
print(f"Image Embedding Diversity: {final_report['quality_assessment']['image_diversity']}")

# Add retrieval performance if available
if 'error' not in text_retrieval_results:
    print("\n🔍 TEXT RETRIEVAL PERFORMANCE:")
    for k in [1, 5, 10]:
        if k in text_retrieval_results['precision_at_k']:
            print(f"  Precision@{k}: {text_retrieval_results['precision_at_k'][k]:.4f}")

print("\n" + "=" * 60)

## 12. Key Metrics Summary and Recommendations

In [ ]:
# Create final visualization comparing both models
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Metrics to compare
metrics = ['Dimension', 'Mean Similarity', 'Std Similarity', 'Effective Rank']
text_values = [
    text_eval_results['embedding_stats']['dimension'],
    text_eval_results['similarity_stats']['mean_similarity'],
    text_eval_results['similarity_stats']['std_similarity'],
    text_eval_results['diversity_stats']['effective_rank']
]
image_values = [
    image_eval_results['embedding_stats']['dimension'],
    image_eval_results['similarity_stats']['mean_similarity'],
    image_eval_results['similarity_stats']['std_similarity'],
    image_eval_results['diversity_stats']['effective_rank']
]

# 1. Dimension comparison
axes[0, 0].bar(['Text Model', 'Image Model'], [text_values[0], image_values[0]], 
               color=['skyblue', 'lightcoral'], alpha=0.7)
axes[0, 0].set_title('Embedding Dimensions')
axes[0, 0].set_ylabel('Dimension')

# 2. Mean Similarity comparison
axes[0, 1].bar(['Text Model', 'Image Model'], [text_values[1], image_values[1]], 
               color=['skyblue', 'lightcoral'], alpha=0.7)
axes[0, 1].set_title('Mean Cosine Similarity')
axes[0, 1].set_ylabel('Mean Similarity')

# 3. Similarity Standard Deviation comparison
axes[0, 2].bar(['Text Model', 'Image Model'], [text_values[2], image_values[2]], 
               color=['skyblue', 'lightcoral'], alpha=0.7)
axes[0, 2].set_title('Similarity Standard Deviation')
axes[0, 2].set_ylabel('Std Similarity')

# 4. Effective Rank comparison
axes[1, 0].bar(['Text Model', 'Image Model'], [text_values[3], image_values[3]], 
               color=['skyblue', 'lightcoral'], alpha=0.7)
axes[1, 0].set_title('Effective Rank (Diversity)')
axes[1, 0].set_ylabel('Effective Rank')

# 5. Text vs Image similarity distributions
text_similarities = text_eval_results['similarity_matrix'][np.triu_indices_from(text_eval_results['similarity_matrix'], k=1)]
image_similarities = image_eval_results['similarity_matrix'][np.triu_indices_from(image_eval_results['similarity_matrix'], k=1)]

axes[1, 1].hist(text_similarities, bins=30, alpha=0.7, label='Text', color='skyblue', density=True)
axes[1, 1].hist(image_similarities, bins=30, alpha=0.7, label='Image', color='lightcoral', density=True)
axes[1, 1].set_title('Similarity Distributions Comparison')
axes[1, 1].set_xlabel('Cosine Similarity')
axes[1, 1].set_ylabel('Density')
axes[1, 1].legend()

# 6. Model performance summary
performance_data = [
    ['Text Model', 'hiieu/halong_embedding', f"{text_values[0]}", f"{text_values[1]:.3f}", final_report['quality_assessment']['text_model_quality']],
    ['Image Model', 'dinov2_vitl14', f"{image_values[0]}", f"{image_values[1]:.3f}", final_report['quality_assessment']['image_model_quality']]
]

axes[1, 2].axis('tight')
axes[1, 2].axis('off')
table = axes[1, 2].table(cellText=performance_data,
                        colLabels=['Model', 'Name', 'Dimension', 'Mean Sim', 'Quality'],
                        cellLoc='center',
                        loc='center')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1.2, 2)
axes[1, 2].set_title('Model Performance Summary')

plt.tight_layout()
plt.show()

print("\n🏆 EVALUATION COMPLETE!")
print("This comprehensive evaluation provides insights into both embedding models' performance.")
print("Use these metrics to monitor model quality and identify potential improvements.")

## 13. Evaluation Metrics Explanation

### Key Metrics Used:

#### **1. Cosine Similarity Distribution**
- **What it measures**: How similar embeddings are to each other
- **Good range**: Mean similarity 0.3-0.7, Standard deviation > 0.1
- **Interpretation**: Higher std indicates better discrimination between different items

#### **2. Effective Rank**
- **What it measures**: Diversity of embeddings (rank/dimension ratio)
- **Good range**: > 0.8 (high diversity), 0.5-0.8 (moderate), < 0.5 (low diversity)
- **Interpretation**: Higher values indicate embeddings use the full embedding space

#### **3. Embedding Norm Distribution**  
- **What it measures**: Consistency of embedding magnitudes
- **Good indicator**: Small standard deviation relative to mean
- **Interpretation**: Consistent norms indicate stable embedding generation

#### **4. Silhouette Score** (when available)
- **What it measures**: Quality of clustering in embedding space
- **Good range**: > 0.5 (good), 0.3-0.5 (fair), < 0.3 (poor)
- **Interpretation**: Higher scores indicate better natural groupings

#### **5. Precision@K / Recall@K** (for text retrieval)
- **What it measures**: Accuracy of finding relevant items in top-K results
- **Good range**: > 0.8 for Precision@1, > 0.6 for Precision@5
- **Interpretation**: Higher values indicate better retrieval performance

### **Model-Specific Insights:**

#### **Text Model (hiieu/halong_embedding)**:
- Optimized for Vietnamese text understanding
- Used for finding similar products based on descriptions
- Input: Product descriptions from existing inventory

#### **Image Model (dinov2_vitl14)**:
- Self-supervised vision transformer
- Used for visual similarity search
- Input: User-uploaded images (base64 encoded)

This evaluation framework can be run regularly to monitor model performance and detect degradation over time.